# RetinaScreen AI: PyTorch to TensorFlow Migration

Training EfficientNetV2S on Original and CLAHE datasets.

In [ ]:
import os, json, glob
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

TRAIN_ORIGINAL = '/content/drive/MyDrive/DATASETS/Diabetic_Retinopathy_dataset/train/original'
TEST_ORIGINAL = '/content/drive/MyDrive/DATASETS/Diabetic_Retinopathy_dataset/test/original'

TRAIN_CLAHE = '/content/drive/MyDrive/DATASETS/Diabetic_Retinopathy_dataset/train/CLAHE'
TEST_CLAHE = '/content/drive/MyDrive/DATASETS/Diabetic_Retinopathy_dataset/test/CLAHE'

# Output directory for all saved models and configs so they persist in Drive
MODELS_OUTPUT_DIR = '/content/drive/MyDrive/DATASETS/models_output'
os.makedirs(MODELS_OUTPUT_DIR, exist_ok=True)
print("Setup complete.")

## 3. Dataset Verification

In [ ]:
def verify_dataset(path):
    if not os.path.exists(path):
        print(f'Path not found: {path}')
        return
    classes = sorted(os.listdir(path))
    print(f'Classes in {path}: {classes}')
    total = 0
    for c in classes:
        num = len(glob.glob(os.path.join(path, c, '*.*')))
        print(f'  {c}: {num} images')
        total += num
    print(f'  Total: {total}\n')

verify_dataset(TRAIN_ORIGINAL)
verify_dataset(TRAIN_CLAHE)
verify_dataset(TEST_ORIGINAL)
verify_dataset(TEST_CLAHE)

## 4. Train/Validation/Test Handling

In [ ]:
def get_datasets(train_dir, test_dir):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, validation_split=0.2, subset="training", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, validation_split=0.2, subset="validation", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
    )
    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir, seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
    )
    return train_ds, val_ds, test_ds

orig_train, orig_val, orig_test = get_datasets(TRAIN_ORIGINAL, TEST_ORIGINAL)
clahe_train, clahe_val, clahe_test = get_datasets(TRAIN_CLAHE, TEST_CLAHE)

CLASS_NAMES = orig_train.class_names
NUM_CLASSES = len(CLASS_NAMES)
print("Class Names:", CLASS_NAMES)

## 6. Model Architecture

In [ ]:
def build_model():
    base_model = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base_model.trainable = False # Freeze backbone
    
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    # Augmentation can be added here
    x = tf.keras.applications.efficientnet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Recall(name='recall')])
    return model

## 5. Experiments

In [ ]:
def run_experiment(name, train_ds, val_ds):
    print(f"\n--- Running Experiment: {name} ---")
    model = build_model()
    
    model_save_path = os.path.join(MODELS_OUTPUT_DIR, f'efficientnetv2s_{name.lower()}_best.keras')
    
    callbacks_list = [
        callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        callbacks.ModelCheckpoint(model_save_path, save_best_only=True)
    ]
    
    # Phase 1: Train Head
    model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks_list)
    
    # Phase 2: Fine-tune
    model.layers[2].trainable = True # unfreeze backbone
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks_list)
    return model

# orig_model = run_experiment("Original", orig_train, orig_val)
# clahe_model = run_experiment("CLAHE", clahe_train, clahe_val)

## 11. Evaluation & Artifacts

In [ ]:
def evaluate_model(model, test_ds, name):
    print(f"\n--- Evaluating {name} ---")
    y_true = []
    y_pred = []
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
    
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    return f1_score(y_true, y_pred, average='macro')

# eval_orig = evaluate_model(orig_model, orig_test, "Original Model")
# eval_clahe = evaluate_model(clahe_model, clahe_test, "CLAHE Model")

def save_artifacts():
    with open(os.path.join(MODELS_OUTPUT_DIR, "class_names.json"), "w") as f:
        json.dump(CLASS_NAMES, f)
        
    base_config = {
        "image_size": IMG_SIZE,
        "class_names": CLASS_NAMES,
        "class_mapping": {c: i for i, c in enumerate(CLASS_NAMES)},
        "num_classes": NUM_CLASSES,
        "output_activation": "softmax",
        "target_layer": "top_conv"
    }
    
    orig_config = base_config.copy()
    orig_config["model_name"] = "efficientnetv2s_original"
    orig_config["use_clahe"] = False
    
    clahe_config = base_config.copy()
    clahe_config["model_name"] = "efficientnetv2s_clahe"
    clahe_config["use_clahe"] = True
    
    with open(os.path.join(MODELS_OUTPUT_DIR, "original_config.json"), "w") as f:
        json.dump(orig_config, f)
    with open(os.path.join(MODELS_OUTPUT_DIR, "clahe_config.json"), "w") as f:
        json.dump(clahe_config, f)
    print("Artifacts saved to Drive.")

# save_artifacts()